In [1]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [ ]:
from post_processing_functions import load_traffic_centers, process_all_regions, save_results

def run_flood_analysis_multi_buffer():
    # Configuration
    traffic_centers_file = r"P:\bovenregionale-stresstest-hwn\Data\Traffic_centrals\Traffic_centers.xlsx"
    #hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Data\Hazard_maps\Hazard_maps-in_use" #change this to add regions from analysis
    hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Analysis"
    output_directory = r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis"
    region_list = ["ARK-NZK", "Vallei en Veluwe", "Noord-Westelijke Delta", "Noord-Brabant Oost"]
    buffer_distances = [5, 10, 50, 100, 200]  # meters

    all_summaries = {}

    for buffer_distance in buffer_distances:
        print(f"\n=== Running analysis for buffer distance: {buffer_distance}m ===")
        # Step 1: Load and buffer traffic centers
        gdf_buffered = load_traffic_centers(traffic_centers_file, buffer_distance)

        # Step 2: Process all regions and flood maps
        gdf_with_floods, all_results = process_all_regions(gdf_buffered, region_list, hazard_maps_base_path)

        # Step 3: Save results in a subfolder for each buffer distance
        buffer_output_dir = f"{output_directory}\\buffer_{buffer_distance}m"
        gpkg_path, excel_path = save_results(gdf_with_floods, buffer_output_dir)

        # Generate summary statistics
        summary_stats = {
            'total_traffic_centers': len(gdf_with_floods),
            'centers_with_flood_data': (gdf_with_floods['max_flood_depth'] != -9999).sum(),
            'centers_with_flooding': (gdf_with_floods['max_flood_depth'] > 0).sum(),
            'max_flood_depth_found': gdf_with_floods['max_flood_depth'].max(),
            'mean_flood_depth': gdf_with_floods[gdf_with_floods['max_flood_depth'] != -9999]['max_flood_depth'].mean(),
            'gpkg_path': gpkg_path,
            'excel_path': excel_path
        }
        all_summaries[buffer_distance] = summary_stats

        print(f"\nAnalysis complete for buffer {buffer_distance}m!")
        print(f"Processed {len(all_results)} flood maps across {len(region_list)} regions.")

    return all_summaries

summaries = run_flood_analysis_multi_buffer()

In [6]:
region_list = ["Vallei en Veluwe", "Noord-Westelijke Delta","Achterhoek", "Brabantse Delta","Friesland","Groningen en NO-Drenthe","Limburg","ARK-NZK",  "Noord-Brabant Oost",
               "Rivierenland","Scheldestromen"
               ]

In [2]:
region_list = ["Noord-Westelijke Delta"]

In [ ]:
#On off ramp analysis
from pathlib import Path
import geopandas as gpd
from post_processing_functions import cluster_connected,aggregate_clusters_to_points
#region_list = ["Friesland", "Vallei en Veluwe","Noord-Westelijke Delta","Limburg"]


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    network_file = root_dir / "base_network_hazard.gpkg"
    network_gdf = gpd.read_file(network_file)
    ramps_gdf = network_gdf[network_gdf["BST_CODE_N"].isin(["AFR", "OPR"])]
    afr_gdf = ramps_gdf[ramps_gdf["BST_CODE_N"] == "AFR"].copy()
    opr_gdf = ramps_gdf[ramps_gdf["BST_CODE_N"] == "OPR"].copy()
    
    afr_gdf_clustered = cluster_connected(afr_gdf)
    output_gpkg = root_dir / "afr_LineSegments.gpkg"
    afr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    
    afr_gdf_aggregated = aggregate_clusters_to_points(afr_gdf_clustered, "EV1_ma", method="mean")
    output_gpkg_aggregated = root_dir / "afr_Points.gpkg"
    afr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")

    opr_gdf_clustered = cluster_connected(opr_gdf)
    output_gpkg = root_dir / "opr_LineSegments.gpkg"
    opr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    opr_gdf_aggregated = aggregate_clusters_to_points(opr_gdf_clustered, "EV1_ma", method="mean")
    output_gpkg_aggregated = root_dir / "opr_Points.gpkg"
    opr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")


In [3]:
from post_processing_functions import Thresholding_for_artefacts_ver02,Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized
import pandas as pd 
# add tunnels and bridge % columns to exposure and damage files and filter

data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("filtered_tunnels.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

kunstinweg_gdf = gpd.read_file(kunstinweg)
kunstinweg_gdf['geometry'] = kunstinweg_gdf['geometry'].buffer(0.2) # to make sure lines are valid
kunstinweg_gdf.rename(columns={'OMSCHR': 'objecttekst'}, inplace=True)

kunstinweg_bridge = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'brug']
kunstinweg_tunnel = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'tunnel']


tunnels_gdf_kunstoverweg = gpd.read_file(tunnels)
bridges_gdf_kunstoverweg = gpd.read_file(bridges) 

bridges_gdf = gpd.GeoDataFrame(
    pd.concat([bridges_gdf_kunstoverweg, kunstinweg_bridge], ignore_index=True),
    crs=bridges_gdf_kunstoverweg.crs
)

tunnels_gdf = gpd.GeoDataFrame(
    pd.concat([tunnels_gdf_kunstoverweg, kunstinweg_tunnel], ignore_index=True),
    crs=tunnels_gdf_kunstoverweg.crs
)



# Define allowed values for bridges and tunnels (lowercased for case-insensitive matching)
allowed_bridges = [
    'aanbrug', 'brug', 'brug (beweegbaar)', 'brug (landbouw)', 'brug (vast)',
    'brug beton', 'brug beton in', 'brug beton over', 'brug beweegbaar',
    'brug hout in', 'brug in', 'brug in de toerit va', 'brug staal in',
    'brug vast', 'vaste brug'
]

allowed_tunnels = [
    'cervedict tunnel', 'open tunnelbak', 'tunnel', 'tunnel vlak',
    'tunnelbak', 'tunnelbak den kaat'
]

# Convert to lowercase for case-insensitive comparison
allowed_bridges = [x.lower() for x in allowed_bridges]
allowed_tunnels = [x.lower() for x in allowed_tunnels]

# Filter bridges
filtered_gdf_brug = bridges_gdf[
    bridges_gdf['objecttekst'].str.lower().isin(allowed_bridges)
]

# Filter tunnels
filtered_gdf_tunnel_and_bridges = tunnels_gdf[
    tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels + allowed_bridges)
]




In [4]:
from post_processing_functions import Thresholding_for_artefacts_ver02,Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized

# add tunnels and bridge % columns to exposure and damage files and filter
'''
data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("filtered_tunnels.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

tunnels_gdf = gpd.read_file(tunnels)
bridges_gdf = gpd.read_file(bridges)   
filtered_gdf_tunnel_and_bridges = tunnels_gdf[tunnels_gdf['objecttekst'].str.contains('tunnel|brug', case=False, na=False)]
filtered_gdf_brug = bridges_gdf[bridges_gdf['objecttekst'].str.contains('brug', case=False, na=False)]
'''

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "base_network_hazard.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)
    roads_dm = root_dir / "damages\HZ_damage_segmented.gpkg"
    roads_dm_gdf = gpd.read_file(roads_dm)


    points_gdf = gpd.read_file(road_height_path)
    points_gdf = points_gdf.to_crs(roads_ex_gdf.crs)
    roads_ex_gdf['Z_height'] = get_z_height_optimized(roads_ex_gdf, points_gdf, threshold=50.0)
    roads_dm_gdf['Z_height'] = get_z_height_optimized(roads_dm_gdf, points_gdf, threshold=50.0)

    print("Calculating tunnel and bridge percentages...")
    Roads_exposure = calculate_overlay_percentages(roads_ex_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges)
    Roads_damage = calculate_overlay_percentages(roads_dm_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges)

    #Roads_exposure.to_file(root_dir / "base_network_hazard_overlay.gpkg", driver='GPKG')
    #Roads_damage.to_file(root_dir / "damages\HZ_damage_segmented_overlay.gpkg", driver='GPKG')
    print("Applying thresholding to remove artefacts...")
    Thresholding_for_artefacts_ver02(Roads_exposure, root_dir)
    

    print("Filtering and aggregating flooded segments...")
    #Filter_and_aggregate_flooded_segments_exposure(Roads_exposure, root_dir,"", dissolve_col='NETWERKSCH')
    #Filter_and_aggregate_flooded_segments_damage(Roads_damage, root_dir,"dam_", dissolve_col='NETWERKSCH')



Processing region: Noord-Westelijke Delta network
Calculating tunnel and bridge percentages...
Applying thresholding to remove artefacts...
Filtering and aggregating flooded segments...
